# Kayfa Content Marketing Crew — Testing Notebook (CrewAI)

**Purpose:** validate the 3-agent pipeline logic in a notebook sandbox before moving to the full VS Code / production folder structure.

**Pipeline:**
```
topic, brand
   -> Strategist-Researcher  (plans + researches, grounded in Tavily + Product Catalog)
   -> Writer-Editor          (writes, self-edits, fact-checks against sources)
   -> 🛑 HUMAN APPROVAL GATE  (simulated here with input())
   -> Publisher              (PDF + email + social captions with hashtags/bullets)
```

**What's mocked in this sandbox** (so you can test without full infra):
- Product catalog -> in-memory list instead of live MongoDB Atlas
- Email sending -> printed, not actually sent (flip `ENABLE_EMAIL_SEND` to go live)
- Ollama -> optional; falls back to a cheap OpenAI/Groq model if no local server is running

Improvements over the reference notebook this was built from: multi-provider LLM routing per agent, native `Knowledge` sources instead of ad-hoc strings, `output_pydantic` instead of `output_json` (current CrewAI convention), a real human-approval gate between drafting and publishing, and AgentOps cost/latency monitoring wired in from the start.


In [1]:
!pip install -qU "crewai[tools]" litellm agentops tavily-python fpdf2 nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Imports

In [2]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from fpdf import FPDF

import os
import json
import agentops
import nest_asyncio

nest_asyncio.apply()  # patches Colab/Jupyter's running event loop so kickoff_async() works cleanly


## 2. API Keys & AgentOps Monitoring

Set these as Colab secrets (`userdata`) or plain environment variables, whichever this environment supports. AgentOps gives you cost/token/latency tracking for free — dashboard link prints after `agentops.init()`.

In [3]:
# --- Set these 4 secrets in Colab's secrets manager (key icon, left sidebar), then toggle notebook access on ---
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["AGENTOPS_API_KEY"] = userdata.get("AGENTOPS_API_KEY")

agentops.init(
    api_key=os.environ["AGENTOPS_API_KEY"],
    skip_auto_end_session=True,
    default_tags=["crewai", "kayfa-content-crew"],
)


## 3. Multi-LLM Setup — cost-aware routing per agent

Match model cost to task complexity: real reasoning gets a frontier model, repetitive/formatting work gets the cheapest option available.

In [4]:
strategist_llm = LLM(model="gpt-4o-mini", temperature=0.3)          # planning needs real judgment
writer_llm     = LLM(model="gpt-4o-mini", temperature=0.5)          # user-facing quality matters
publisher_llm  = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)  # fast, cheap, formatting-only

# Optional local Ollama swap for the Publisher (uncomment if you have a local server running):
# publisher_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434")

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
output_dir = "./kayfa_crew_output"
os.makedirs(output_dir, exist_ok=True)


## 4. Company Knowledge (simulated brand + product facts)

In production these live as files in `knowledge/` (see file structure discussed earlier). Here they're inlined as `StringKnowledgeSource` for fast notebook iteration.

In [5]:
brand_voice_guide = StringKnowledgeSource(content="""
Kayfa (\u0643\u064a\u0641) is a bilingual Arabic/English e-learning platform.
Tone: warm, encouraging, practical, never corporate. Avoid words like
'leverage', 'synergy', 'disrupt'. Prefer 'use', 'learn', 'grow', 'practical'.
""")

company_info = StringKnowledgeSource(content="""
Kayfa offers courses in both Arabic and English, built for both languages
from the ground up (not translated). Focus areas: business skills, tech
skills, personal development. Audience: career-changers and working
professionals seeking practical skills.
""")

approved_claims = StringKnowledgeSource(content="""
Approved claims only: 'Kayfa offers courses in both Arabic and English.'
'Kayfa focuses on practical, applicable skills.' Never state specific
user counts, revenue, or growth percentages unless explicitly provided.
""")

technical_style_guide = StringKnowledgeSource(content="""
Technical writing rules (used only when content_type='technical_writing'):
Prioritize accuracy over persuasion -- no hashtags, no CTA, no hype language.
Structure: prerequisites list, then numbered steps, each with an expected
result. Include a troubleshooting note for common failure points. Code/config
values must be exact and copy-pasteable. Neutral, instructional tone --
this is documentation, not a growth post.
""")


## 5. Tools — search, brand-voice check, fact verification, PDF, email, char-limit

In [6]:
@tool("web_search")
def web_search(query: str) -> str:
    """Live web search for current facts, trends, and competitor content. Input: a query string."""
    return json.dumps(tavily_client.search(query=query, max_results=5))


@tool("brand_voice_check")
def brand_voice_check(draft_text: str) -> str:
    """Flags off-brand words in a draft against Kayfa's tone guide. Input: draft_text.
    Output: list of flagged words/phrases found, empty if none."""
    banned = ["leverage", "synergy", "disrupt", "revolutionize", "utilize"]
    flagged = [w for w in banned if w in draft_text.lower()]
    return json.dumps({"flagged_words": flagged, "clean": len(flagged) == 0})


@tool("verify_source_claim")
def verify_source_claim(claim: str, source_hint: str) -> str:
    """Re-searches a claim to confirm it's still supported by current web results,
    instead of trusting the writer's citation blindly. Input: claim, source_hint (topic/keyword)."""
    results = tavily_client.search(query=f"{claim} {source_hint}", max_results=3)
    supported = len(results.get("results", [])) > 0
    return json.dumps({"claim": claim, "supported_by_search": supported})


ENABLE_EMAIL_SEND = False  # flip to True once real SMTP creds are wired in

@tool("send_email_with_pdf")
def send_email_with_pdf(recipient: str, subject: str, pdf_path: str) -> str:
    """Emails the generated PDF to a recipient. Input: recipient, subject, pdf_path.
    In sandbox mode (ENABLE_EMAIL_SEND=False) this only prints instead of sending."""
    if not ENABLE_EMAIL_SEND:
        return f"[SANDBOX] Would email '{subject}' with attachment {pdf_path} to {recipient}"
    # Real implementation: wire up smtplib or an email API here, using env-var credentials.
    return f"Email sent to {recipient} with attachment {pdf_path}"


@tool("validate_char_limit")
def validate_char_limit(text: str, platform: str) -> str:
    """Hard-validates text length against platform limits. Input: text, platform ('x' or 'linkedin').
    Never trust an LLM to count characters itself."""
    limits = {"x": 280, "linkedin": 3000}
    limit = limits.get(platform.lower(), 3000)
    return json.dumps({"valid": len(text) <= limit, "char_count": len(text), "limit": limit})


## 7. Pydantic Output Schemas

`output_pydantic` guarantees a validated, typed object back from each task — the "content contract" for this pipeline.

In [7]:
class ContentPlan(BaseModel):
    audience: str
    angle: str
    outline: List[str] = Field(default_factory=list, description="3-6 point outline, guidance only, not enforced")
    target_keywords: List[str] = Field(default_factory=list, description="3-8 target keywords, guidance only, not enforced")
    sources: List[str] = Field(default_factory=list, description="URLs used for research")


class PostSection(BaseModel):
    heading: str
    body: str = Field(..., description="Can include markdown bullet points, e.g. '- point one'")


class BlogPost(BaseModel):
    content_type: str = Field(default="marketing_blog", description="'marketing_blog' or 'technical_writing'")
    title: str
    slug: str
    meta_description: str
    key_takeaways: List[str] = Field(default_factory=list, description="Scannable bullet summary at the top, aim for 3-5")
    intro: str
    sections: List[PostSection] = Field(default_factory=list, description="Used for marketing_blog content")
    prerequisites: List[str] = Field(default_factory=list, description="Technical writing only -- what the reader needs before starting")
    steps: List[PostSection] = Field(default_factory=list, description="Technical writing only -- numbered steps, heading=step title, body=instructions+expected result")
    conclusion: str
    cta: str = Field(default="", description="Marketing only -- leave empty for technical_writing")
    hashtags: List[str] = Field(default_factory=list, description="Marketing only -- 5-8 hashtags, leave empty for technical_writing")
    word_count: int = 0
    sources: List[str] = Field(default_factory=list)


class SocialOutput(BaseModel):
    linkedin_post: str
    tweets: List[str] = Field(default_factory=list)
    hashtags: List[str] = Field(default_factory=list)


## 8. Agent 1 — Strategist-Researcher (plans + researches in one pass)

In [8]:
strategist_researcher = Agent(
    role="Content Strategist & Researcher",
    goal="Plan a content angle and gather verified, current facts before any writing begins.",
    backstory=(
        "A senior content lead at Kayfa who understands the bilingual "
        "Arabic/English e-learning audience deeply, and never lets a plan "
        "move forward on unverified assumptions."
    ),
    llm=strategist_llm,
    tools=[web_search],
    allow_delegation=False,
    verbose=True,
)

plan_research_task = Task(
    description="\n".join([
        "Topic: {topic}. Brand: {brand}.",
        "1. Define the target audience and content angle.",
        "2. Produce a 3-6 point outline.",
        "3. Research 3-8 target keywords using web_search.",
        "4. List every source URL you actually used.",
    ]),
    expected_output="A structured content plan with audience, angle, outline, keywords, and sources.",
    output_pydantic=ContentPlan,
    output_file=os.path.join(output_dir, "step_1_content_plan.json"),
    agent=strategist_researcher,
)


## 9. Agent 2 — Writer-Editor (writes, self-edits, fact-checks in one pass)

In [9]:
writer_editor = Agent(
    role="Writer-Editor",
    goal=(
        "Write a complete piece of content following the content plan, adapting tone "
        "and structure to the requested content_type (marketing_blog or technical_writing), "
        "then self-edit and verify every factual claim before finalizing."
    ),
    backstory=(
        "A bilingual writer at Kayfa who can switch between persuasive marketing copy "
        "and precise technical documentation depending on the assignment, but never "
        "compromises on brand voice or factual accuracy in either mode."
    ),
    llm=writer_llm,
    tools=[brand_voice_check, verify_source_claim],
    allow_delegation=False,
    verbose=True,
)


def build_write_edit_task(content_type: str = "marketing_blog") -> Task:
    """Builds the Writer-Editor task with instructions branched by content_type.
    Same agent, same schema -- only the description and expected_output change."""

    if content_type == "technical_writing":
        instructions = [
            "Using the content plan, write a TECHNICAL GUIDE (content_type='technical_writing').",
            "List prerequisites the reader needs before starting.",
            "Write numbered 'steps' (not 'sections'): each step's heading is the step title, "
            "body is the instruction + the expected result, using exact copy-pasteable values.",
            "Prioritize accuracy over persuasion -- no hashtags, no CTA, no hype language.",
            "Add a short troubleshooting note for the most likely failure point.",
            "Run brand_voice_check on your draft anyway -- Kayfa tone still applies to explanations, "
            "just without marketing language.",
            "Run verify_source_claim on every factual/technical claim before finalizing.",
            "Leave 'cta' and 'hashtags' empty; leave 'sections' empty and use 'steps' instead.",
        ]
        expected = "A complete, accurate, brand-checked technical guide with prerequisites and numbered steps."
    else:
        instructions = [
            "Using the content plan, write a complete 800-1000 word MARKETING BLOG POST (content_type='marketing_blog').",
            "Include a 'key_takeaways' bullet list summarizing the post up top.",
            "Use bullet points inside sections wherever there's a sequence, comparison, or list of features.",
            "Run brand_voice_check on your draft and rewrite any flagged phrases.",
            "Run verify_source_claim on every factual claim before finalizing.",
            "Generate 5-8 hashtags: 2-3 broad e-learning tags, 2-3 topic-specific tags, 1-2 branded (#Kayfa).",
            "Leave 'prerequisites' and 'steps' empty; use 'sections' instead.",
        ]
        expected = "A complete, brand-checked, fact-verified marketing blog post."

    return Task(
        description="\n".join(instructions),
        expected_output=expected,
        output_pydantic=BlogPost,
        output_file=os.path.join(output_dir, "step_2_blog_post.json"),
        agent=writer_editor,
        context=[plan_research_task],
    )


## 10. Run the Draft Stage (Strategist-Researcher -> Writer-Editor)

This is the notebook equivalent of the `/generate` FastAPI endpoint from the production design.

In [10]:
# Switch this to "technical_writing" to generate a how-to guide instead of a marketing post
CONTENT_TYPE = "marketing_blog"  # or "technical_writing"

write_edit_task = build_write_edit_task(CONTENT_TYPE)

draft_crew = Crew(
    agents=[strategist_researcher, writer_editor],
    tasks=[plan_research_task, write_edit_task],
    process=Process.sequential,
    memory=True,
    knowledge_sources=[brand_voice_guide, company_info, approved_claims, technical_style_guide],
    verbose=True,
)

draft_result = await draft_crew.kickoff_async(inputs={
    "topic": "AI agents for small businesses",
    "brand": "Kayfa",
    "content_type": CONTENT_TYPE,
})

blog_post: BlogPost = draft_result.pydantic
print(blog_post.model_dump_json(indent=2))


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3597a2c8-418a-4971-bdd4-377d198965eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Topic: AI agents for small businesses. Brand: Kayfa.                                                     │
│  1. Define the target audience and content angle.                                                               │
│  2. Produce a 3-6 point outline.                                                                                │
│  3. Research 3-8 target keywords using web_search.                                                              │
│  4. List every source URL you actually used.                                                                    │
│  ID: 2be42dc5-9dc4-40a2-8c7b-f8ba44c498db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 2584.38ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the content plan, write a complete 800-1000 word MARKETING BLOG POST                               │
│  (content_type='marketing_blog').                                                                               │
│  Include a 'key_takeaways' bullet list summarizing the post up top.                                             │
│  Use bullet points inside sections wherever there's a sequence, comparison, or list of features.                │
│  Run brand_voice_check on your draft and rewrite any flagged phrases.                                           │
│  Run verify_source_claim on every factual claim before finalizing.                                              │
│  Generate 5-8 hashtags: 2-3 broad e-learning tags, 2-3 topic-specific tags, 1-2 branded (#Kayfa).               │
│  Leave 'prerequisites' and 'steps' empty; use 'sections' instead.                                               │
│  Agent: Writer-Editor                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3597a2c8-418a-4971-bdd4-377d198965eb                                                                       │
│  Final Output: {                                                                                                │
│    "content_type": "marketing_blog",                                                                            │
│    "title": "Transform Your Small Business with AI Agents",                                                     │
│    "slug": "transform-small-business-ai-agents",                                                                │
│    "meta_description": "Discover how AI agents can automate tasks, improve efficiency, and provide              │
│  cost-effective solutions for small businesses in 2026.",                                                       │
│    "key_takeaways": [                                                                                           │
│      "AI agents are transforming small businesses by automating tasks and improving efficiency.",               │
│      "They provide cost-effective solutions, making them ideal for entrepreneurs.",                             │
│      "Choosing the right AI agent can significantly enhance productivity."                                      │
│    ],                                                                                                           │
│    "intro": "In the fast-paced world of small business, staying ahead of the competition is crucial. AI agents  │
│  are emerging as a game-changer, offering innovative solutions to streamline operations. These intelligent      │
│  systems can handle repetitive tasks, allowing entrepreneurs to focus on growth and customer engagement.",      │
│    "sections": [                                                                                                │
│      {                                                                                                          │
│        "heading": "Benefits of Using AI Agents",                                                                │
│        "body": "Integrating AI agents into your small business can lead to several advantages:\n- **Cost        │
│  Savings**: Automating tasks reduces the need for additional staff, lowering operational costs.\n-              │
│  **Efficiency**: AI agents can perform tasks faster and more accurately than humans, increasing overall         │
│  productivity.\n- **Scalability**: As your business grows, AI solutions can easily adapt to changing needs      │
│  without significant investment."                                                                               │
│      },                                                                                                         │
│      {                                                                                                          │
│        "heading": "Top AI Agents for Small Businesses in 2026",                                                 │
│        "body": "Here are some of the best AI agents available for small businesses this year:\n1.               │
│  **Chatbots**: Enhance customer service by providing instant responses to inquiries.\n2. **Virtual              │
│  Assistants**: Manage schedules, emails, and reminders, freeing up valuable time.\n3. **Predictive Analytics    │
│  Tools**: Help in making data-driven decisions by forecasting trends.\n4. **Content Creation Tools**: Generate  │
│  marketing content quickly and efficiently.\n5. **Sale

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 3351.90ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
  "content_type": "marketing_blog",
  "title": "Transform Your Small Business with AI Agents",
  "slug": "transform-small-business-ai-agents",
  "meta_description": "Discover how AI agents can automate tasks, improve efficiency, and provide cost-effective solutions for small businesses in 2026.",
  "key_takeaways": [
    "AI agents are transforming small businesses by automating tasks and improving efficiency.",
    "They provide cost-effective solutions, making them ideal for entrepreneurs.",
    "Choosing the right AI agent can significantly enhance productivity."
  ],
  "intro": "In the fast-paced world of small business, staying ahead of the competition is crucial. AI agents are emerging as a game-changer, offering innovative solutions to streamline operations. These intelligent systems can handle repetitive tasks, allowing entrepreneurs to focus on growth and customer engagement.",
  "sections": [
    {
      "heading": "Benefits of Using AI Agents",
      "body": "Integrating A

## 11. 🛑 Human-in-the-Loop Approval Gate (simulated)

In production this is the split between `/generate` and `/approve` in FastAPI. In this notebook sandbox, `input()` stands in for that human click.

In [11]:
print("=" * 60)
print(f"DRAFT READY FOR REVIEW ({blog_post.content_type}): {blog_post.title}")
print("=" * 60)
if blog_post.content_type == "technical_writing":
    print(f"Prerequisites: {blog_post.prerequisites}")
    print(f"Steps: {[s.heading for s in blog_post.steps]}")
else:
    print(f"Key takeaways: {blog_post.key_takeaways}")
    print(f"Hashtags: {blog_post.hashtags}")
print(f"Word count: {blog_post.word_count}")
print("=" * 60)

approval = input("Approve this post for publishing? (y/n): ").strip().lower()
human_approved = approval == "y"
print("APPROVED" if human_approved else "REJECTED -- stopping before Publisher stage")


DRAFT READY FOR REVIEW (marketing_blog): Transform Your Small Business with AI Agents
Key takeaways: ['AI agents are transforming small businesses by automating tasks and improving efficiency.', 'They provide cost-effective solutions, making them ideal for entrepreneurs.', 'Choosing the right AI agent can significantly enhance productivity.']
Hashtags: ['#eLearning', '#SmallBusiness', '#AI', '#Productivity', '#Entrepreneurs', '#Kayfa']
Word count: 817
Approve this post for publishing? (y/n): y
APPROVED


## 12. Agent 3 — Publisher (PDF + email + social captions, runs ONLY if approved)

In [12]:
publisher = Agent(
    role="Publisher",
    goal="Turn an approved post into platform-ready social captions with hashtags and bullets.",
    backstory="A distribution specialist who formats and delivers already-approved content, never edits substance.",
    llm=publisher_llm,
    tools=[validate_char_limit],
    allow_delegation=False,
    verbose=True,
)

repurpose_task = Task(
    description="\n".join([
        "Using the approved blog post, write a LinkedIn post (use bullet points and hashtags)",
        "and 2 tweets (use validate_char_limit to confirm each is under 280 characters).",
        "Reuse the same hashtags as the blog post where relevant.",
    ]),
    expected_output="LinkedIn post and 2 tweets, all within platform character limits.",
    output_pydantic=SocialOutput,
    output_file=os.path.join(output_dir, "step_3_social_output.json"),
    agent=publisher,
)


In [15]:
import os
import textwrap
from fpdf import FPDF
# Import the explicit position Enums for modern fpdf2 compatibility
from fpdf.enums import XPos, YPos

# =====================================================================
# 1. CRITICAL HOTFIX: Prevents Groq 'cache_breakpoint' API rejection
# =====================================================================
try:
    import crewai.llms.cache as _crewai_cache
    _crewai_cache.mark_cache_breakpoint = lambda msg: msg
except ImportError:
    # Fallback in case CrewAI structure updates in future library versions
    pass

# =====================================================================
# 2. UPDATED PDF UTILITIES
# =====================================================================

def _pdf_safe(text: str, width: int = 90) -> str:
    """Two safety layers for fpdf2's core fonts:
    1. Latin-1 sanitize -- Helvetica can't render Arabic/Unicode, replace with '?'.
    2. Hard-wrap at `width` chars, force-breaking any unbroken run (long URL, slug,
       or a run of replaced '?' chars with no spaces) so fpdf2 always finds a line break.
    """
    safe = text.encode("latin-1", errors="replace").decode("latin-1")
    wrapped_lines = []
    for line in safe.split("\n"):
        if not line.strip():
            wrapped_lines.append(line)
            continue
        wrapped_lines.extend(
            textwrap.wrap(line, width=width, break_long_words=True, break_on_hyphens=False) or [""]
        )
    return "\n".join(wrapped_lines)


def _cell(pdf, h, text):
    """multi_cell wrapper updated for modern fpdf2 compatibility.
    Using explicit Enums ensures the cursor moves predictably to the
    left margin on a new line, preventing 'out of horizontal space' errors.
    """
    # Calculate usable width dynamically (Page Width - Left Margin - Right Margin)
    usable_width = pdf.epw

    # Use modern Enums for x/y cursor behavior
    pdf.multi_cell(
        w=usable_width,
        h=h,
        text=_pdf_safe(text),
        new_x=XPos.LMARGIN,
        new_y=YPos.NEXT
    )


def generate_pdf(post, path: str) -> str:
    """Renders the approved BlogPost into a simple formatted PDF."""
    pdf = FPDF()
    pdf.add_page()

    # Title
    pdf.set_font("Helvetica", "B", 16)
    _cell(pdf, 10, post.title)

    # Meta
    pdf.set_font("Helvetica", "", 11)
    _cell(pdf, 8, f"Meta: {post.meta_description}\n")

    if post.content_type == "technical_writing":
        pdf.set_font("Helvetica", "B", 12)
        _cell(pdf, 8, "Prerequisites:")
        pdf.set_font("Helvetica", "", 11)
        for item in post.prerequisites:
            _cell(pdf, 7, f"  - {item}")

        _cell(pdf, 8, "\n" + post.intro + "\n")

        for i, step in enumerate(post.steps, start=1):
            pdf.set_font("Helvetica", "B", 12)
            _cell(pdf, 8, f"Step {i}: {step.heading}")
            pdf.set_font("Helvetica", "", 11)
            _cell(pdf, 7, step.body + "\n")

        pdf.set_font("Helvetica", "", 11)
        _cell(pdf, 7, post.conclusion)
    else:
        pdf.set_font("Helvetica", "B", 12)
        _cell(pdf, 8, "Key Takeaways:")
        pdf.set_font("Helvetica", "", 11)
        for point in post.key_takeaways:
            _cell(pdf, 7, f"  - {point}")

        _cell(pdf, 8, "\n" + post.intro + "\n")

        for section in post.sections:
            pdf.set_font("Helvetica", "B", 12)
            _cell(pdf, 8, section.heading)
            pdf.set_font("Helvetica", "", 11)
            _cell(pdf, 7, section.body + "\n")

        pdf.set_font("Helvetica", "", 11)
        _cell(pdf, 7, post.conclusion + "\n" + post.cta)
        _cell(pdf, 7, " ".join(f"#{h.lstrip('#')}" for h in post.hashtags))

    # Ensure output directory exists before saving
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pdf.output(path)
    return path

## 13. Close AgentOps Session

Check the printed dashboard link for cost, token usage, latency, and tool-call breakdowns per agent.

In [16]:
agentops.end_session("Success" if human_approved else "Indeterminate")


🖇 AgentOps: end_session() is deprecated and will be removed in v4 in the future. Use agentops.end_trace() instead.
🖇 AgentOps: end_session called but no active trace context found.
